# `demografia.ipynb` - Extracción demográfica resiliente (API + fallback)

Requisito: *"Si la API REST no está disponible o falla la conexión HTTP, el script no debe detenerse. Debes implementar un mecanismo secundario de respaldo interno."*

`obtener_datos_demograficos` intenta la llamada HTTP hasta `API_MAX_REINTENTOS + 1` veces. Se contemplan **dos formas de fallo**, no solo la excepción de red:

1. **Fallo de conexión / HTTP**: timeout, DNS, código de estado de error (`raise_for_status()` lanza excepción) -> se captura `requests.exceptions.RequestException`.
2. **Fallo "silencioso"**: la API responde `200 OK` pero con un cuerpo que **no tiene el formato esperado** (por ejemplo un endpoint deprecado que devuelve `{"success": false, "data": null, ...}` en vez de la lista de países). Esto se detecta validando la forma del JSON, no solo el código HTTP.

Si tras todos los reintentos no se ha conseguido una respuesta válida, se activa `POBLACION_FALLBACK` (de `config.ipynb`) y el pipeline continúa sin interrupciones.

> **Nota**: al preparar este proyecto se comprobó que el endpoint `restcountries.com/v3.1/...` indicado en la especificación **está actualmente deprecado** por el proveedor. En la práctica, el mecanismo de fallback se activa siempre al ejecutar este pipeline hoy — la celda de prueba de más abajo lo demuestra con datos reales.

> Depende de `config.ipynb`, `logging_config.ipynb` (necesita la variable `logger`) y `transform.ipynb` (usa `normalizar_nombre_pais`).

In [ ]:
import logging

import requests

## Función principal

In [ ]:
def obtener_datos_demograficos():
    """
    Intenta obtener {PAIS_NORMALIZADO: poblacion} desde la API REST.
    Si la API no responde, responde con un cuerpo inesperado, o falla
    la conexión HTTP, cae automáticamente al diccionario de respaldo
    interno (POBLACION_FALLBACK) para que el pipeline pueda continuar.

    Devuelve una tupla (diccionario_poblacion, origen) donde origen
    es 'API' o 'FALLBACK'.
    """
    intentos = 0
    while intentos <= API_MAX_REINTENTOS:
        intentos += 1
        try:
            logger.info(
                f"Intento {intentos}/{API_MAX_REINTENTOS + 1} de conexion a la API "
                f"de datos demograficos: {API_DEMOGRAFIA_URL}"
            )
            respuesta = requests.get(API_DEMOGRAFIA_URL, timeout=API_TIMEOUT_SEGUNDOS)
            respuesta.raise_for_status()
            cuerpo = respuesta.json()

            # La API puede devolver 200 OK pero con un cuerpo de error
            # (p. ej. {"success": false, "data": null, ...}) en vez de
            # la lista de países esperada. Se valida la forma real de
            # la respuesta, no solo el código HTTP.
            if not isinstance(cuerpo, list) or len(cuerpo) == 0:
                raise ValueError(
                    "La API respondio 200 OK pero el cuerpo no tiene el formato "
                    "esperado (lista de paises) - posible deprecacion de la API."
                )

            poblacion_api = {}
            for pais_api in cuerpo:
                nombre_comun = pais_api.get("name", {}).get("common")
                poblacion = pais_api.get("population")
                if nombre_comun and poblacion is not None:
                    clave = normalizar_nombre_pais(nombre_comun)
                    poblacion_api[clave] = poblacion

            if not poblacion_api:
                raise ValueError("La API respondio correctamente pero sin datos usables.")

            logger.info(
                f"Datos demograficos obtenidos correctamente desde la API "
                f"({len(poblacion_api)} paises)."
            )
            return poblacion_api, "API"

        except (requests.exceptions.RequestException, ValueError) as error:
            logger.warning(f"Fallo al obtener datos demograficos de la API: {error}")

    # Mecanismo secundario de respaldo interno (requisito de resiliencia)
    logger.warning(
        "No fue posible obtener datos demograficos de la API tras "
        f"{API_MAX_REINTENTOS + 1} intentos. Se activa el dataset de "
        "respaldo interno (fallback) para no detener la ejecucion."
    )
    return dict(POBLACION_FALLBACK), "FALLBACK"

## Prueba rápida

Llamada real a la función, para ver en vivo los `WARNING` (si la API falla) y confirmar el origen final de los datos (`API` o `FALLBACK`).

In [ ]:
poblacion_prueba, origen_prueba = obtener_datos_demograficos()

print(f"\nOrigen: {origen_prueba}")
print(f"Paises disponibles: {len(poblacion_prueba)}")
print("Ejemplo:", {k: poblacion_prueba[k] for k in list(poblacion_prueba)[:3]})